# 03 — Neural Network from Scratch (with Manual Backpropagation)

**Goal:** build a neural network from raw NumPy — forward propagation, and critically, **backpropagation by hand**. This is the part most people skip even after finishing an ML course, because frameworks like TensorFlow do it for you with `.fit()`. Here, you compute every gradient yourself.

**Why backprop is broken into small pieces in this notebook:** backprop is really just the chain rule applied repeatedly. If you implement it as one big function, it's easy to get a working-but-not-understood result. Instead, each link in the chain gets its own cell and its own sanity check, so you feel every step rather than jumping straight to a final formula.

**Numerical gradient checking:** after building analytical backprop, you'll implement a completely different way of estimating gradients (tiny nudges to each parameter, measuring the resulting change in cost) and compare it against your backprop output. If they match closely, your backprop is correct — independent of anything I tell you the "expected value" should be. This is a real technique working engineers use to debug neural networks, and it means you'll leave this notebook able to verify your own backprop in the future without needing anyone to hand you expected values.

**Architecture:** input → ReLU hidden layer → sigmoid output layer (binary classification). We extend to 2 hidden layers near the end, once the pattern is solid with one.

**Structure:**
1. Activation functions + their derivatives
2. Forward propagation (1 hidden layer)
3. Cost function
4. Backpropagation, broken into small chain-rule steps
5. Numerical gradient checking
6. Full training loop on a non-linearly-separable dataset
7. Extend to 2 hidden layers
8. Reference solutions


In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)


---
## Part 1 — Activation Functions and Their Derivatives

You need both the function AND its derivative for backprop — the derivative is what lets gradients flow backward through each activation.

### ReLU
$$\text{ReLU}(z) = \max(0, z)$$
$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

### Sigmoid (you've already implemented this in Lab 2 — reimplement it here so this notebook is self-contained)
$$\sigma(z) = \frac{1}{1+e^{-z}}$$
$$\sigma'(z) = \sigma(z)(1-\sigma(z))$$


In [5]:
def relu(z):
    """
    Compute ReLU activation.

    Args:
        z (ndarray): input, any shape

    Returns:
        a (ndarray): ReLU(z), same shape as z
    """
    # TODO: implement
    return (z > 0).astype(float)


In [ ]:
def relu_derivative(z):
    """
    Compute derivative of ReLU at z.

    Args:
        z (ndarray): input, any shape (the PRE-activation values, not the activated output)

    Returns:
        d (ndarray): derivative of ReLU at each point in z, same shape as z (values are 0 or 1)
    """
    # TODO: implement
    # hint: (z > 0) is a boolean array; casting it to a numeric type gives you 0s and 1s
    # lemme correct the relu derivative just  now 
    
    pass


In [ ]:
def sigmoid(z):
    """
    Compute sigmoid activation.

    Args:
        z (ndarray or scalar): input

    Returns:
        a (ndarray or scalar): sigmoid(z), values in (0,1)
    """
    # TODO: implement
    # adding the sigmoid no w
    pass


In [6]:
# Sanity checks
z_test = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])

r = relu(z_test)
print("relu:", r)
print("expected: [0. 0. 0. 0.5 2.]")
assert np.allclose(r, [0., 0., 0., 0.5, 2.])

rd = relu_derivative(z_test)
print("\nrelu_derivative:", rd)
print("expected: [0. 0. 0. 1. 1.]")
assert np.allclose(rd, [0., 0., 0., 1., 1.])

s = sigmoid(np.array([0.0]))
print("\nsigmoid(0):", s)
assert np.isclose(s[0], 0.5)

print("\nAll activation functions passed!")


relu: [0. 0. 0. 1. 1.]
expected: [0. 0. 0. 0.5 2.]


AssertionError: 

---
## Part 2 — Forward Propagation (1 Hidden Layer)

Architecture: input layer ($n$ features) → hidden layer ($h$ units, ReLU) → output layer (1 unit, sigmoid).

For a batch of $m$ examples stacked as rows in $X$ (shape `(m, n)`):

$$Z^{[1]} = X W^{[1]} + b^{[1]}, \quad A^{[1]} = \text{ReLU}(Z^{[1]})$$
$$Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}, \quad A^{[2]} = \sigma(Z^{[2]})$$

Shapes (this is the part people get wrong most often — get comfortable with it now):
- $X$: `(m, n)` — m examples, n input features
- $W^{[1]}$: `(n, h)` — maps n inputs to h hidden units
- $b^{[1]}$: `(h,)` — one bias per hidden unit
- $Z^{[1]}, A^{[1]}$: `(m, h)`
- $W^{[2]}$: `(h, 1)` — maps h hidden units to 1 output
- $b^{[2]}$: `(1,)` or scalar
- $Z^{[2]}, A^{[2]}$: `(m, 1)`

$A^{[2]}$ is the final prediction — a probability, just like logistic regression's output.


In [ ]:
def forward_propagation(X, W1, b1, W2, b2):
    """
    Compute forward pass through a 1-hidden-layer network (ReLU hidden, sigmoid output).

    Args:
        X (ndarray (m,n)): input data, m examples, n features
        W1 (ndarray (n,h)): hidden layer weights
        b1 (ndarray (h,)): hidden layer biases
        W2 (ndarray (h,1)): output layer weights
        b2 (ndarray (1,)): output layer bias

    Returns:
        A2 (ndarray (m,1)): final predictions (probabilities)
        cache (dict): stores Z1, A1, Z2, A2 -- you will need ALL of these for backprop later
    """
    # TODO: implement
    # hint: cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    pass


In [ ]:
# Sanity check with tiny hand-checkable network: 2 inputs, 2 hidden units, 1 output
X_test = np.array([[1.0, 2.0]])   # 1 example, 2 features
W1_test = np.array([[0.1, -0.2], [0.3, 0.4]])   # shape (2,2)
b1_test = np.array([0.0, 0.0])
W2_test = np.array([[0.5], [-0.5]])              # shape (2,1)
b2_test = np.array([0.0])

A2, cache = forward_propagation(X_test, W1_test, b1_test, W2_test, b2_test)

# Manual computation to verify:
# Z1 = [1*0.1 + 2*0.3, 1*(-0.2) + 2*0.4] = [0.7, 0.6]
# A1 = relu([0.7, 0.6]) = [0.7, 0.6]  (both positive, unchanged)
# Z2 = 0.7*0.5 + 0.6*(-0.5) = 0.35 - 0.3 = 0.05
# A2 = sigmoid(0.05)

expected_A2 = sigmoid(np.array([[0.05]]))
print("Your A2:", A2)
print("Expected A2:", expected_A2)
assert np.allclose(A2, expected_A2, atol=1e-6)

print("Your cached Z1:", cache["Z1"])
print("Expected Z1:   [[0.7 0.6]]")
assert np.allclose(cache["Z1"], [[0.7, 0.6]])

print("\nforward_propagation passed!")


### 2.1 — Cost function

Same binary cross-entropy (log loss) as logistic regression — the output layer IS logistic regression, just fed engineered features from the hidden layer instead of raw inputs.

$$J = -\frac{1}{m}\sum_{i=0}^{m-1}\Big[y^{(i)}\log(a^{(i)}) + (1-y^{(i)})\log(1-a^{(i)})\Big]$$


In [ ]:
def compute_cost(A2, y):
    """
    Compute log loss (binary cross-entropy) cost.

    Args:
        A2 (ndarray (m,1)): predicted probabilities from forward_propagation
        y (ndarray (m,) or (m,1)): true binary labels

    Returns:
        cost (scalar): average log loss
    """
    # TODO: implement
    # hint: reshape y to (m,1) if needed with y.reshape(-1,1), and clip A2 with
    # np.clip(A2, 1e-10, 1 - 1e-10) before taking the log, same as in Lab 2
    pass


In [ ]:
# Sanity check
y_test = np.array([1])
cost = compute_cost(A2, y_test)
expected_cost = -np.log(expected_A2[0,0])
print("Your cost:", cost)
print("Expected cost:", expected_cost)
assert np.isclose(cost, expected_cost, atol=1e-6)
print("compute_cost passed!")


---
## Part 3 — Backpropagation, One Chain-Rule Link at a Time

This is the core of the lab. Backprop computes $\frac{\partial J}{\partial W^{[1]}}, \frac{\partial J}{\partial b^{[1]}}, \frac{\partial J}{\partial W^{[2]}}, \frac{\partial J}{\partial b^{[2]}}$ — how much each parameter should change to reduce cost.

The trick: you never compute these directly. You compute them **backward**, one layer at a time, reusing intermediate results — that's literally what "back-propagation" means. Each step below is one link in that chain.

We'll use a small batch (m=3) so you can verify each intermediate value by hand if you want to.


In [ ]:
# Small dataset for backprop derivation, m=3 examples, 2 features
X_bp = np.array([
    [1.0, 2.0],
    [-1.0, 0.5],
    [2.0, -1.0]
])
y_bp = np.array([1, 0, 1])

W1_bp = np.array([[0.1, -0.2], [0.3, 0.4]])
b1_bp = np.array([0.0, 0.0])
W2_bp = np.array([[0.5], [-0.5]])
b2_bp = np.array([0.0])

A2_bp, cache_bp = forward_propagation(X_bp, W1_bp, b1_bp, W2_bp, b2_bp)
print("A2 (predictions):", A2_bp.ravel())


### 3.1 — $\frac{\partial J}{\partial A^{[2]}}$ (dA2)

Start at the very end: how does cost change if the final prediction $A^{[2]}$ changes slightly?

For binary cross-entropy:
$$\frac{\partial J}{\partial A^{[2]}} = \frac{1}{m}\left(-\frac{y}{A^{[2]}} + \frac{1-y}{1-A^{[2]}}\right)$$

This is the very first link in the chain — nothing to combine yet, just this formula directly.


In [ ]:
def compute_dA2(A2, y):
    """
    Compute dJ/dA2, the gradient of cost with respect to the final activation.

    Args:
        A2 (ndarray (m,1)): predicted probabilities
        y (ndarray (m,)): true labels

    Returns:
        dA2 (ndarray (m,1)): gradient of cost w.r.t. A2
    """
    # TODO: implement
    # hint: reshape y to (m,1) with y.reshape(-1,1); clip A2 same as in compute_cost
    # to avoid dividing by 0
    pass


In [ ]:
# Sanity check
dA2 = compute_dA2(A2_bp, y_bp)
m = X_bp.shape[0]
y_col = y_bp.reshape(-1,1)
A2_clipped = np.clip(A2_bp, 1e-10, 1-1e-10)
expected_dA2 = (1/m) * (-y_col/A2_clipped + (1-y_col)/(1-A2_clipped))

print("Your dA2:", dA2.ravel())
print("Expected:", expected_dA2.ravel())
assert np.allclose(dA2, expected_dA2, atol=1e-6)
print("compute_dA2 passed!")


### 3.2 — $\frac{\partial J}{\partial Z^{[2]}}$ (dZ2)

Chain rule link: $\frac{\partial J}{\partial Z^{[2]}} = \frac{\partial J}{\partial A^{[2]}} \cdot \frac{\partial A^{[2]}}{\partial Z^{[2]}}$

$\frac{\partial A^{[2]}}{\partial Z^{[2]}}$ is the sigmoid derivative, $\sigma(Z^{[2]})(1-\sigma(Z^{[2]})) = A^{[2]}(1-A^{[2]})$ (since $A^{[2]}$ already IS $\sigma(Z^{[2]})$).

**Neat fact worth noticing:** when you multiply these two pieces together, the algebra simplifies enormously — this is the exact same simplification that made logistic regression's gradient look clean in Lab 2. Combining sigmoid + log loss like this is not a coincidence.

$$dZ^{[2]} = \frac{A^{[2]} - y}{m}$$

(The $\frac{1}{m}$ here comes along from `dA2`, which already has it baked in.) You can derive this by multiplying your `dA2` by the sigmoid derivative and simplifying — try it on paper if you want to see the cancellation — but for the code, implement it either way (multiply dA2 by sigmoid derivative, OR directly as (A2-y)/m) and the sanity check will confirm they agree.


In [ ]:
def compute_dZ2(dA2, A2):
    """
    Compute dJ/dZ2 by combining dA2 with the sigmoid derivative (chain rule).

    Args:
        dA2 (ndarray (m,1)): gradient of cost w.r.t. A2 (from compute_dA2)
        A2 (ndarray (m,1)): the activation itself (sigmoid output)

    Returns:
        dZ2 (ndarray (m,1)): gradient of cost w.r.t. Z2
    """
    # TODO: implement
    # hint: sigmoid derivative in terms of its own output is A2 * (1 - A2)
    # dZ2 = dA2 * (that derivative)
    pass


In [ ]:
# Sanity check -- this should also equal (A2 - y)/m after simplification, verify both ways
dZ2 = compute_dZ2(dA2, A2_bp)
m = X_bp.shape[0]
expected_dZ2 = (A2_bp - y_col) / m   # the simplified closed form (note the 1/m factor carried over from dA2)

print("Your dZ2:   ", dZ2.ravel())
print("Expected:   ", expected_dZ2.ravel())
assert np.allclose(dZ2, expected_dZ2, atol=1e-6)
print("compute_dZ2 passed! (notice it matches the simplified (A2 - y)/m form)")


### 3.3 — $\frac{\partial J}{\partial W^{[2]}}$ and $\frac{\partial J}{\partial b^{[2]}}$ (dW2, db2)

Now branch off from `dZ2` to get the gradients for the actual output-layer parameters:

$$dW^{[2]} = (A^{[1]})^T dZ^{[2]}$$
$$db^{[2]} = \sum dZ^{[2]}$$

**No extra $\frac{1}{m}$ here** — `dZ2` already carries the averaging factor, since it was inherited from `dA2` (which had `1/m` baked in from the start). Dividing by `m` again here would double-count it. This is one of the easiest places to introduce a subtle bug in backprop, and exactly the kind of mistake the numerical gradient checker in Part 4 is designed to catch.

(This mirrors the vectorized gradient pattern from your linear/logistic regression labs — $X^T \cdot \text{error}$ — just with $A^{[1]}$ playing the role $X$ played before, since $A^{[1]}$ is literally the "input" to this layer.)


In [ ]:
def compute_dW2_db2(dZ2, A1):
    """
    Compute gradients for the output layer's weights and bias.

    Args:
        dZ2 (ndarray (m,1)): gradient of cost w.r.t. Z2 (already includes the 1/m averaging factor)
        A1 (ndarray (m,h)): hidden layer activations (from forward prop cache)

    Returns:
        dW2 (ndarray (h,1)): gradient of cost w.r.t. W2
        db2 (ndarray (1,)): gradient of cost w.r.t. b2
    """
    # TODO: implement
    # hint: dZ2 already has the 1/m factor baked in (inherited from dA2) -- do NOT divide by m again here
    pass


In [ ]:
# Sanity check
dW2, db2 = compute_dW2_db2(dZ2, cache_bp["A1"])
expected_dW2 = cache_bp["A1"].T @ dZ2
expected_db2 = np.sum(dZ2, axis=0)

print("Your dW2:", dW2.ravel())
print("Expected:", expected_dW2.ravel())
assert np.allclose(dW2, expected_dW2, atol=1e-6)

print("\nYour db2:", db2)
print("Expected:", expected_db2)
assert np.allclose(db2, expected_db2, atol=1e-6)
print("\ncompute_dW2_db2 passed!")


### 3.4 — $\frac{\partial J}{\partial A^{[1]}}$ (dA1)

Now propagate the error backward INTO the hidden layer. How does cost change if the hidden layer's activation changes? Push `dZ2` back through $W^{[2]}$:

$$dA^{[1]} = dZ^{[2]} (W^{[2]})^T$$

This is the step that makes it "back"-propagation — you're using $W^{[2]}$ (a forward-pass parameter) to send the error signal backward.


In [ ]:
def compute_dA1(dZ2, W2):
    """
    Propagate the gradient backward from the output layer into the hidden layer's activations.

    Args:
        dZ2 (ndarray (m,1)): gradient of cost w.r.t. Z2
        W2 (ndarray (h,1)): output layer weights

    Returns:
        dA1 (ndarray (m,h)): gradient of cost w.r.t. A1
    """
    # TODO: implement
    pass


In [ ]:
# Sanity check
dA1 = compute_dA1(dZ2, W2_bp)
expected_dA1 = dZ2 @ W2_bp.T

print("Your dA1:\n", dA1)
print("Expected:\n", expected_dA1)
assert np.allclose(dA1, expected_dA1, atol=1e-6)
print("\ncompute_dA1 passed!")


### 3.5 — $\frac{\partial J}{\partial Z^{[1]}}$ (dZ1)

Same pattern as 3.2, but now through the ReLU derivative instead of sigmoid's:

$$dZ^{[1]} = dA^{[1]} \odot \text{ReLU}'(Z^{[1]})$$

($\odot$ means elementwise multiplication, not matrix multiplication — this is a key difference from the matrix multiplies used elsewhere.)


In [ ]:
def compute_dZ1(dA1, Z1):
    """
    Combine dA1 with the ReLU derivative (chain rule) to get dZ1.

    Args:
        dA1 (ndarray (m,h)): gradient of cost w.r.t. A1
        Z1 (ndarray (m,h)): pre-activation values from the hidden layer (from forward prop cache)

    Returns:
        dZ1 (ndarray (m,h)): gradient of cost w.r.t. Z1
    """
    # TODO: implement
    # hint: elementwise multiply dA1 by relu_derivative(Z1) -- use * not @
    pass


In [ ]:
# Sanity check
dZ1 = compute_dZ1(dA1, cache_bp["Z1"])
expected_dZ1 = dA1 * relu_derivative(cache_bp["Z1"])

print("Your dZ1:\n", dZ1)
print("Expected:\n", expected_dZ1)
assert np.allclose(dZ1, expected_dZ1, atol=1e-6)
print("\ncompute_dZ1 passed!")


### 3.6 — $\frac{\partial J}{\partial W^{[1]}}$ and $\frac{\partial J}{\partial b^{[1]}}$ (dW1, db1)

Same pattern as 3.3, one layer earlier. $X$ plays the role $A^{[1]}$ played before, since $X$ is literally the "input" to this layer.

$$dW^{[1]} = X^T dZ^{[1]}$$
$$db^{[1]} = \sum dZ^{[1]}$$

Same note as before: no extra $\frac{1}{m}$ here — it's already baked into `dZ1`, carried all the way from `dA2` at the very start of the chain.


In [ ]:
def compute_dW1_db1(dZ1, X):
    """
    Compute gradients for the hidden layer's weights and bias.

    Args:
        dZ1 (ndarray (m,h)): gradient of cost w.r.t. Z1 (already includes the 1/m averaging factor)
        X (ndarray (m,n)): the original input data

    Returns:
        dW1 (ndarray (n,h)): gradient of cost w.r.t. W1
        db1 (ndarray (h,)): gradient of cost w.r.t. b1
    """
    # TODO: implement
    # hint: same as compute_dW2_db2 -- no /m needed, it's already in dZ1
    pass


In [ ]:
# Sanity check
dW1, db1 = compute_dW1_db1(dZ1, X_bp)
expected_dW1 = X_bp.T @ dZ1
expected_db1 = np.sum(dZ1, axis=0)

print("Your dW1:\n", dW1)
print("Expected:\n", expected_dW1)
assert np.allclose(dW1, expected_dW1, atol=1e-6)

print("\nYour db1:", db1)
print("Expected:", expected_db1)
assert np.allclose(db1, expected_db1, atol=1e-6)
print("\ncompute_dW1_db1 passed!")


### 3.7 — Full backward_propagation: chain everything together

Now combine all six pieces above into one function. This should just be six function calls in sequence, in the exact order you built them.


In [ ]:
def backward_propagation(X, y, W2, cache):
    """
    Run the full backward pass, computing gradients for all parameters.

    Args:
        X (ndarray (m,n)): input data
        y (ndarray (m,)): true labels
        W2 (ndarray (h,1)): output layer weights (needed to propagate error backward)
        cache (dict): Z1, A1, Z2, A2 from forward_propagation

    Returns:
        grads (dict): dictionary with keys "dW1", "db1", "dW2", "db2"
    """
    # TODO: implement by calling, in order:
    # compute_dA2 -> compute_dZ2 -> compute_dW2_db2
    #             -> compute_dA1 -> compute_dZ1 -> compute_dW1_db1
    # then package results into a dict
    pass


In [ ]:
# Sanity check -- should reproduce the same dW1,db1,dW2,db2 you already verified piece by piece
grads = backward_propagation(X_bp, y_bp, W2_bp, cache_bp)

assert np.allclose(grads["dW2"], dW2, atol=1e-6)
assert np.allclose(grads["db2"], db2, atol=1e-6)
assert np.allclose(grads["dW1"], dW1, atol=1e-6)
assert np.allclose(grads["db1"], db1, atol=1e-6)
print("backward_propagation passed! Full chain reproduces the piece-by-piece results.")


---
## Part 4 — Numerical Gradient Checking

Here's a completely independent way to estimate a gradient, with no chain rule and no backprop at all: **nudge one parameter by a tiny amount, and see how much the cost changes.**

$$\frac{\partial J}{\partial \theta} \approx \frac{J(\theta + \epsilon) - J(\theta - \epsilon)}{2\epsilon}$$

This is slow (you'd have to do it separately for every single parameter, one at a time) and impractical for real training — but it's an excellent way to check whether your backprop is actually correct, since it doesn't depend on your backprop code being right at all.

We'll implement it for just `W1` (looping over each entry), and compare against your analytical `dW1` from backprop. If they match closely, you've independently verified your entire backprop chain is correct.


In [ ]:
def numerical_gradient_check(X, y, W1, b1, W2, b2, epsilon=1e-4):
    """
    Numerically estimate dJ/dW1 by nudging each entry of W1 up and down by epsilon,
    and compare against the analytical dW1 from backward_propagation.

    Args:
        X (ndarray (m,n)): input data
        y (ndarray (m,)): true labels
        W1, b1, W2, b2: current parameters
        epsilon (float): small perturbation size

    Returns:
        numerical_dW1 (ndarray, same shape as W1): numerically estimated gradient
    """
    numerical_dW1 = np.zeros_like(W1)
    # TODO: implement
    # hint: loop over every index (i,j) in W1 with nested for loops (or np.ndindex(W1.shape))
    # for each entry:
    #   1. make a copy of W1, add epsilon to just that one entry, run forward_propagation + compute_cost -> cost_plus
    #   2. make another copy of W1, subtract epsilon from that same entry, run forward_propagation + compute_cost -> cost_minus
    #   3. numerical_dW1[i,j] = (cost_plus - cost_minus) / (2 * epsilon)
    pass


In [ ]:
# Run the check
numerical_dW1 = numerical_gradient_check(X_bp, y_bp, W1_bp, b1_bp, W2_bp, b2_bp)

A2_check, cache_check = forward_propagation(X_bp, W1_bp, b1_bp, W2_bp, b2_bp)
grads_check = backward_propagation(X_bp, y_bp, W2_bp, cache_check)
analytical_dW1 = grads_check["dW1"]

print("Numerical dW1:\n", numerical_dW1)
print("\nAnalytical dW1 (from your backprop):\n", analytical_dW1)

# relative error is the standard way to compare -- handles different magnitudes better than raw difference
diff = np.linalg.norm(numerical_dW1 - analytical_dW1)
norm_sum = np.linalg.norm(numerical_dW1) + np.linalg.norm(analytical_dW1)
relative_error = diff / norm_sum

print(f"\nRelative error: {relative_error:.2e}")
print("Expected: relative error should be less than 1e-6 (very small)")
assert relative_error < 1e-6, "Gradient check failed -- backprop and numerical gradient disagree, there's likely a bug"
print("\nGRADIENT CHECK PASSED. Your backprop implementation is verified correct.")


**This is a genuinely important milestone.** You've now independently proven your own backprop code is mathematically correct, using a completely different method than the one you used to build it. This is exactly the kind of check real ML engineers use when debugging a new architecture, before trusting a model's training results.


---
## Part 5 — Full Training on a Non-Linearly-Separable Dataset

Time to see why we needed a neural network at all. We'll use `make_moons` — two interleaving crescent shapes that **no straight line can separate**. Logistic regression (Lab 2) would fail badly here. A neural network with a hidden layer can bend its decision boundary to fit this.


In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap='bwr', edgecolor='k', alpha=0.7)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Two moons -- NOT linearly separable")
plt.show()


### 5.1 — Parameter initialization

One important detail: **never initialize weights to all zeros** for a network with a hidden layer. If every hidden unit starts identical, they'll all compute the same gradient and update identically forever -- the network can never break the symmetry and use its hidden units differently from each other. Initialize randomly instead, scaled small so activations don't start too extreme.


In [ ]:
def initialize_parameters(n_x, n_h, n_y=1):
    """
    Randomly initialize network parameters.

    Args:
        n_x (int): number of input features
        n_h (int): number of hidden units
        n_y (int): number of output units (1 for binary classification)

    Returns:
        W1 (ndarray (n_x,n_h)), b1 (ndarray (n_h,)), W2 (ndarray (n_h,n_y)), b2 (ndarray (n_y,))
    """
    # TODO: implement
    # hint: use np.random.randn(shape) * 0.01 for weights (small random values)
    # biases can start at exactly zero -- that's fine, only WEIGHTS need randomness to break symmetry
    pass


In [ ]:
# Sanity check
np.random.seed(1)
W1_i, b1_i, W2_i, b2_i = initialize_parameters(n_x=2, n_h=4, n_y=1)

print("W1 shape:", W1_i.shape, "expected (2, 4)")
print("b1 shape:", b1_i.shape, "expected (4,)")
print("W2 shape:", W2_i.shape, "expected (4, 1)")
print("b2 shape:", b2_i.shape, "expected (1,)")

assert W1_i.shape == (2,4)
assert b1_i.shape == (4,)
assert W2_i.shape == (4,1)
assert b2_i.shape == (1,)
assert np.allclose(b1_i, 0) and np.allclose(b2_i, 0)
assert not np.allclose(W1_i, 0), "W1 should be randomly initialized, not zeros"
print("\ninitialize_parameters passed!")


### 5.2 — Training loop

Join everything: initialize → forward → cost → backward → update parameters → repeat.

Update rule (same gradient descent pattern as every previous lab):
$$W := W - \alpha \, dW, \quad b := b - \alpha \, db$$


In [ ]:
def train_network(X, y, n_h, alpha, num_iters, print_every=500):
    """
    Train a 1-hidden-layer neural network with gradient descent.

    Args:
        X (ndarray (m,n)): input data
        y (ndarray (m,)): binary labels
        n_h (int): number of hidden units
        alpha (float): learning rate
        num_iters (int): number of training iterations
        print_every (int): print cost every this many iterations

    Returns:
        params (dict): final W1, b1, W2, b2
        cost_history (list): cost at every iteration
    """
    n_x = X.shape[1]
    W1, b1, W2, b2 = initialize_parameters(n_x, n_h, 1)
    cost_history = []

    # TODO: implement the training loop
    # for each iteration:
    #   1. forward_propagation(X, W1, b1, W2, b2) -> A2, cache
    #   2. compute_cost(A2, y) -> cost, append to cost_history
    #   3. backward_propagation(X, y, W2, cache) -> grads
    #   4. update W1, b1, W2, b2 using grads and alpha
    #   (optional) print cost every `print_every` iterations to watch progress

    params = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    return params, cost_history


In [ ]:
# Train on the moons dataset
params, cost_history = train_network(X_moons, y_moons, n_h=8, alpha=0.5, num_iters=5000)

plt.plot(cost_history)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("Training cost -- should flatten out")
plt.show()

print("Final cost:", cost_history[-1])
assert cost_history[-1] < cost_history[0]
assert cost_history[-1] < 0.3, "Cost should be reasonably low on this dataset with a working network"
print("Training completed!")


In [ ]:
# Accuracy + decision boundary visualization
A2_final, _ = forward_propagation(X_moons, params["W1"], params["b1"], params["W2"], params["b2"])
predictions = (A2_final.ravel() >= 0.5).astype(int)
accuracy = np.mean(predictions == y_moons) * 100
print(f"Training accuracy: {accuracy:.2f}%")

xx, yy = np.meshgrid(
    np.linspace(X_moons[:,0].min()-0.5, X_moons[:,0].max()+0.5, 300),
    np.linspace(X_moons[:,1].min()-0.5, X_moons[:,1].max()+0.5, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]
probs, _ = forward_propagation(grid, params["W1"], params["b1"], params["W2"], params["b2"])
probs = probs.reshape(xx.shape)

plt.contourf(xx, yy, probs, levels=[0,0.5,1], colors=['#ffdddd', '#ddddff'], alpha=0.6)
plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap='bwr', edgecolor='k', alpha=0.7)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Neural network decision boundary -- notice it CURVES, unlike logistic regression")
plt.show()


Compare this to your logistic regression decision boundary from Lab 2 -- that one was a straight line by necessity. This one bends to actually follow the two crescents. That curve is entirely a product of the hidden layer + ReLU nonlinearity -- without them, stacking two linear layers would collapse back into just another straight line (worth thinking about why: a linear function of a linear function is still linear).


---
## Part 6 — Extending to 2 Hidden Layers

Now generalize the pattern to a deeper network: input → hidden layer 1 (ReLU) → hidden layer 2 (ReLU) → output (sigmoid).

$$Z^{[1]} = XW^{[1]}+b^{[1]}, A^{[1]}=\text{ReLU}(Z^{[1]})$$
$$Z^{[2]} = A^{[1]}W^{[2]}+b^{[2]}, A^{[2]}=\text{ReLU}(Z^{[2]})$$
$$Z^{[3]} = A^{[2]}W^{[3]}+b^{[3]}, A^{[3]}=\sigma(Z^{[3]})$$

Backprop follows the exact same pattern as before, just one more layer to propagate through -- notice the pattern repeats: `dZ` at each layer always combines the incoming gradient with that layer's activation derivative, and `dA` at the previous layer always comes from pushing the current `dZ` back through that layer's weights.


In [ ]:
def forward_propagation_deep(X, W1, b1, W2, b2, W3, b3):
    """
    Forward pass for a 2-hidden-layer network (ReLU, ReLU, sigmoid).

    Args:
        X (ndarray (m,n)): input data
        W1,b1: layer 1 params (n -> h1)
        W2,b2: layer 2 params (h1 -> h2)
        W3,b3: output layer params (h2 -> 1)

    Returns:
        A3 (ndarray (m,1)): final predictions
        cache (dict): Z1,A1,Z2,A2,Z3,A3
    """
    # TODO: implement, following the same pattern as forward_propagation but with 3 layers
    pass


In [ ]:
def backward_propagation_deep(X, y, W2, W3, cache):
    """
    Backward pass for a 2-hidden-layer network.

    Args:
        X (ndarray (m,n)): input data
        y (ndarray (m,)): true labels
        W2, W3: needed to propagate error backward through layers 2 and 3
        cache (dict): Z1,A1,Z2,A2,Z3,A3 from forward_propagation_deep

    Returns:
        grads (dict): dW1,db1,dW2,db2,dW3,db3
    """
    # TODO: implement
    # hint: this is the SAME pattern as backward_propagation, just one more layer:
    # dA3 (reuse compute_dA2 logic, treating A3 as the final output)
    # -> dZ3 (reuse compute_dZ2 logic -- sigmoid derivative)
    # -> dW3, db3 (reuse compute_dW2_db2 logic, using A2 as the "input" to this layer)
    # -> dA2 (reuse compute_dA1 logic, using W3 to propagate back)
    # -> dZ2 (reuse compute_dZ1 logic -- ReLU derivative, using Z2)
    # -> dW2, db2 (reuse compute_dW1_db1 logic, using A1 as the "input" to this layer)
    # -> dA1 (same propagation pattern again, using W2)
    # -> dZ1 (ReLU derivative, using Z1)
    # -> dW1, db1 (using X as the "input" to this layer)
    pass


In [ ]:
# Sanity check via numerical gradient checking on W2 (the middle layer -- most likely place for a bug)
def initialize_parameters_deep(n_x, n_h1, n_h2, n_y=1):
    np.random.seed(2)
    W1 = np.random.randn(n_x, n_h1) * 0.01
    b1 = np.zeros(n_h1)
    W2 = np.random.randn(n_h1, n_h2) * 0.01
    b2 = np.zeros(n_h2)
    W3 = np.random.randn(n_h2, n_y) * 0.01
    b3 = np.zeros(n_y)
    return W1, b1, W2, b2, W3, b3

X_deep_test = X_bp  # reuse the small 3-example dataset from Part 3
y_deep_test = y_bp

W1d, b1d, W2d, b2d, W3d, b3d = initialize_parameters_deep(n_x=2, n_h1=4, n_h2=3)

A3, cache_deep = forward_propagation_deep(X_deep_test, W1d, b1d, W2d, b2d, W3d, b3d)
grads_deep = backward_propagation_deep(X_deep_test, y_deep_test, W2d, W3d, cache_deep)

# Numerically check dW2 specifically
epsilon = 1e-4
numerical_dW2 = np.zeros_like(W2d)
for i in range(W2d.shape[0]):
    for j in range(W2d.shape[1]):
        W2_plus = W2d.copy(); W2_plus[i,j] += epsilon
        A3_plus, _ = forward_propagation_deep(X_deep_test, W1d, b1d, W2_plus, b2d, W3d, b3d)
        cost_plus = compute_cost(A3_plus, y_deep_test)

        W2_minus = W2d.copy(); W2_minus[i,j] -= epsilon
        A3_minus, _ = forward_propagation_deep(X_deep_test, W1d, b1d, W2_minus, b2d, W3d, b3d)
        cost_minus = compute_cost(A3_minus, y_deep_test)

        numerical_dW2[i,j] = (cost_plus - cost_minus) / (2*epsilon)

diff = np.linalg.norm(numerical_dW2 - grads_deep["dW2"])
norm_sum = np.linalg.norm(numerical_dW2) + np.linalg.norm(grads_deep["dW2"])
relative_error = diff / norm_sum

print("Relative error on dW2:", relative_error)
assert relative_error < 1e-6, "Deep backprop gradient check failed"
print("\n2-hidden-layer backprop verified correct via numerical gradient checking!")


In [ ]:
# Train the deep network on the moons dataset
def train_network_deep(X, y, n_h1, n_h2, alpha, num_iters):
    n_x = X.shape[1]
    W1, b1, W2, b2, W3, b3 = initialize_parameters_deep(n_x, n_h1, n_h2, 1)
    cost_history = []
    for i in range(num_iters):
        A3, cache = forward_propagation_deep(X, W1, b1, W2, b2, W3, b3)
        cost = compute_cost(A3, y)
        cost_history.append(cost)
        grads = backward_propagation_deep(X, y, W2, W3, cache)
        W1 = W1 - alpha * grads["dW1"]
        b1 = b1 - alpha * grads["db1"]
        W2 = W2 - alpha * grads["dW2"]
        b2 = b2 - alpha * grads["db2"]
        W3 = W3 - alpha * grads["dW3"]
        b3 = b3 - alpha * grads["db3"]
    return {"W1":W1,"b1":b1,"W2":W2,"b2":b2,"W3":W3,"b3":b3}, cost_history

params_deep, cost_history_deep = train_network_deep(X_moons, y_moons, n_h1=8, n_h2=6, alpha=0.5, num_iters=5000)

plt.plot(cost_history_deep)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.title("2-hidden-layer network: Cost vs Iteration")
plt.show()

A3_final, _ = forward_propagation_deep(X_moons, params_deep["W1"], params_deep["b1"],
                                          params_deep["W2"], params_deep["b2"],
                                          params_deep["W3"], params_deep["b3"])
predictions_deep = (A3_final.ravel() >= 0.5).astype(int)
accuracy_deep = np.mean(predictions_deep == y_moons) * 100
print(f"Training accuracy (2 hidden layers): {accuracy_deep:.2f}%")


---
## You're done with the core implementation.

You've now built and verified, from raw NumPy:
- Forward propagation for 1 and 2 hidden layers
- Full backpropagation, derived link-by-link through the chain rule
- Numerical gradient checking -- a real debugging technique you can reuse on any future network
- Trained on genuinely non-linearly-separable data and watched the decision boundary bend

**Reference solutions below** -- only look if you're stuck or want to double check your approach after passing all asserts.


---
## Reference Solutions (don't peek until you've passed the asserts above)


In [ ]:
# --- Part 1 reference ---
def relu_ref(z):
    return np.maximum(0, z)

def relu_derivative_ref(z):
    return (z > 0).astype(float)

def sigmoid_ref(z):
    return 1 / (1 + np.exp(-z))


In [ ]:
# --- Part 2 reference ---
def forward_propagation_ref(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu_ref(Z1)
    Z2 = A1 @ W2 + b2
    A2 = sigmoid_ref(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def compute_cost_ref(A2, y):
    y = y.reshape(-1, 1)
    A2c = np.clip(A2, 1e-10, 1 - 1e-10)
    m = y.shape[0]
    return -np.sum(y*np.log(A2c) + (1-y)*np.log(1-A2c)) / m


In [ ]:
# --- Part 3 reference (backprop, piece by piece) ---
def compute_dA2_ref(A2, y):
    y = y.reshape(-1,1)
    m = y.shape[0]
    A2c = np.clip(A2, 1e-10, 1-1e-10)
    return (1/m) * (-y/A2c + (1-y)/(1-A2c))

def compute_dZ2_ref(dA2, A2):
    return dA2 * (A2 * (1 - A2))

def compute_dW2_db2_ref(dZ2, A1):
    # NOTE: no /m here -- dZ2 already carries the 1/m factor inherited from dA2
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0)
    return dW2, db2

def compute_dA1_ref(dZ2, W2):
    return dZ2 @ W2.T

def compute_dZ1_ref(dA1, Z1):
    return dA1 * relu_derivative_ref(Z1)

def compute_dW1_db1_ref(dZ1, X):
    # same note -- no /m here either
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0)
    return dW1, db1

def backward_propagation_ref(X, y, W2, cache):
    dA2 = compute_dA2_ref(cache["A2"], y)
    dZ2 = compute_dZ2_ref(dA2, cache["A2"])
    dW2, db2 = compute_dW2_db2_ref(dZ2, cache["A1"])
    dA1 = compute_dA1_ref(dZ2, W2)
    dZ1 = compute_dZ1_ref(dA1, cache["Z1"])
    dW1, db1 = compute_dW1_db1_ref(dZ1, X)
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


In [ ]:
# --- Part 4 reference ---
def numerical_gradient_check_ref(X, y, W1, b1, W2, b2, epsilon=1e-4):
    numerical_dW1 = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1_plus = W1.copy(); W1_plus[i,j] += epsilon
            A2_plus, _ = forward_propagation_ref(X, W1_plus, b1, W2, b2)
            cost_plus = compute_cost_ref(A2_plus, y)

            W1_minus = W1.copy(); W1_minus[i,j] -= epsilon
            A2_minus, _ = forward_propagation_ref(X, W1_minus, b1, W2, b2)
            cost_minus = compute_cost_ref(A2_minus, y)

            numerical_dW1[i,j] = (cost_plus - cost_minus) / (2*epsilon)
    return numerical_dW1


In [ ]:
# --- Part 5 reference ---
def initialize_parameters_ref(n_x, n_h, n_y=1):
    W1 = np.random.randn(n_x, n_h) * 0.01
    b1 = np.zeros(n_h)
    W2 = np.random.randn(n_h, n_y) * 0.01
    b2 = np.zeros(n_y)
    return W1, b1, W2, b2

def train_network_ref(X, y, n_h, alpha, num_iters, print_every=500):
    n_x = X.shape[1]
    W1, b1, W2, b2 = initialize_parameters_ref(n_x, n_h, 1)
    cost_history = []
    for i in range(num_iters):
        A2, cache = forward_propagation_ref(X, W1, b1, W2, b2)
        cost = compute_cost_ref(A2, y)
        cost_history.append(cost)
        grads = backward_propagation_ref(X, y, W2, cache)
        W1 = W1 - alpha * grads["dW1"]
        b1 = b1 - alpha * grads["db1"]
        W2 = W2 - alpha * grads["dW2"]
        b2 = b2 - alpha * grads["db2"]
    params = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    return params, cost_history


In [ ]:
# --- Part 6 reference (2 hidden layers) ---
def forward_propagation_deep_ref(X, W1, b1, W2, b2, W3, b3):
    Z1 = X @ W1 + b1
    A1 = relu_ref(Z1)
    Z2 = A1 @ W2 + b2
    A2 = relu_ref(Z2)
    Z3 = A2 @ W3 + b3
    A3 = sigmoid_ref(Z3)
    cache = {"Z1":Z1,"A1":A1,"Z2":Z2,"A2":A2,"Z3":Z3,"A3":A3}
    return A3, cache

def backward_propagation_deep_ref(X, y, W2, W3, cache):
    dA3 = compute_dA2_ref(cache["A3"], y)
    dZ3 = compute_dZ2_ref(dA3, cache["A3"])
    dW3, db3 = compute_dW2_db2_ref(dZ3, cache["A2"])

    dA2 = compute_dA1_ref(dZ3, W3)
    dZ2 = compute_dZ1_ref(dA2, cache["Z2"])
    dW2, db2 = compute_dW1_db1_ref(dZ2, cache["A1"])

    dA1 = compute_dA1_ref(dZ2, W2)
    dZ1 = compute_dZ1_ref(dA1, cache["Z1"])
    dW1, db1 = compute_dW1_db1_ref(dZ1, X)

    return {"dW1":dW1,"db1":db1,"dW2":dW2,"db2":db2,"dW3":dW3,"db3":db3}
